In [1]:
from Bio import Phylo, AlignIO, SeqIO, Seq
from Bio.Phylo import TreeConstruction
from Bio.Phylo.TreeConstruction import DistanceTreeConstructor, DistanceCalculator
from Bio.Align.Applications import ClustalwCommandline
import pickle
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm as tqdm
from Bio.SeqRecord import SeqRecord
import umap

/Users/hiroki463/miniforge3/envs/bioinfo/lib/python3.13/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [2]:
results_contigsInfo = pd.read_csv('results32_wgMLST/results_contigsInfo.tsv', sep='\t')
results_contigsInfo['GenomeName'] = results_contigsInfo['FILE']
results_contigsInfo = results_contigsInfo.set_index("FILE")
results_contigsInfo = results_contigsInfo.rename(index={'146_HiFi_consensus':'IFTK146', '189_HiFi_consensus':'IFTK189', 
                                                        '207_HiFi_consensus':'IFTK207', '252_HiFi_consensus':'IFTK252', 
                                                        '345_HiFi_consensus':'IFTK345', '363_HiFi_consensus':'IFTK363',
                                                        '87_HiFi_consensus':'IFTK87', 'ATCC33291_Campylobacter_jejuni_subsp_jejuni':'ATCC33291',
                                                        'ATCC33560_HiFi_consensus':'ATCC33560', 'BB520_HiFi_consensus':'BB520',
                                                        'IA525_HiFi_consensus':'lA525', 'LA525_HiFi_consensus':'LA525',
                                                        'NADC5096_HiFi_consensus':'NADC5096', 'UBB527_HiFi_consensus':'UBB527',
                                                        'ULB527_HiFi_consensus':'ULB527'})
results_contigsInfo

,ATCC33250-protein1,ATCC33250-protein100,ATCC33250-protein1000,ATCC33250-protein1001,ATCC33250-protein1003,ATCC33250-protein1004,ATCC33250-protein1009,ATCC33250-protein1011,ATCC33250-protein1012,ATCC33250-protein1017,...,IFTK60-protein1041,IFTK60-protein1139,IFTK80-protein1502,IFTK93-protein1807,IFTK96-protein1351,IFTK96-protein1354,IFTK96-protein1476,LA525-protein613,UBB527-protein19,GenomeName
FILE,,,,,,,,,,,,,,,,,,,,,
ATCC33250,7bab7a0bf78e48bf_1&1-1323&1,7bab7a0bf78e48bf_1&114167-115672&1,7bab7a0bf78e48bf_1&975508-976275&1,7bab7a0bf78e48bf_1&976407-976871&-1,7bab7a0bf78e48bf_1&980135-980830&-1,7bab7a0bf78e48bf_1&980817-981587&-1,7bab7a0bf78e48bf_1&985828-986262&-1,7bab7a0bf78e48bf_1&986514-987020&-1,7bab7a0bf78e48bf_1&987029-988060&-1,7bab7a0bf78e48bf_1&993056-993631&-1,...,LNF,7bab7a0bf78e48bf_1&1137480-1138643&-1,LNF,LNF,LNF,LNF,LNF,7bab7a0bf78e48bf_1&539841-540281&-1,7bab7a0bf78e48bf_1&25299-26213&-1,ATCC33250
ATCC33291,d37006a1999e4b03_1&1-1323&1,d37006a1999e4b03_1&110992-112497&1,d37006a1999e4b03_1&946108-946875&1,d37006a1999e4b03_1&947007-947471&-1,d37006a1999e4b03_1&950735-951430&-1,d37006a1999e4b03_1&951417-952187&-1,d37006a1999e4b03_1&956428-956862&-1,d37006a1999e4b03_1&957114-957620&-1,d37006a1999e4b03_1&957629-958660&-1,d37006a1999e4b03_1&963654-964229&-1,...,LNF,d37006a1999e4b03_1&1107792-1108955&-1,LNF,LNF,LNF,LNF,d37006a1999e4b03_1&1226159-1226950&1,d37006a1999e4b03_1&549803-550237&-1,d37006a1999e4b03_1&25442-26356&-1,ATCC33291
ATCC33560,cluster_001_consensus&1-1323&1,cluster_001_consensus&113146-114651&1,cluster_001_consensus&939314-940081&1,cluster_001_consensus&1057994-1058458&-1,cluster_001_consensus&1061722-1062417&-1,cluster_001_consensus&1062404-1063174&-1,cluster_001_consensus&1067415-1067849&-1,cluster_001_consensus&1068100-1068606&-1,cluster_001_consensus&1068615-1069646&-1,cluster_001_consensus&1074640-1075215&-1,...,LNF,cluster_001_consensus&1222630-1223793&-1,LNF,LNF,LNF,LNF,LNF,cluster_001_consensus&540204-540644&-1,LNF,ATCC33560
BB520,cluster_001_consensus&1-1323&1,cluster_001_consensus&114128-115633&1,cluster_001_consensus&976754-977521&1,cluster_001_consensus&977653-978117&-1,cluster_001_consensus&981381-982076&-1,cluster_001_consensus&982063-982833&-1,cluster_001_consensus&987074-987508&-1,cluster_001_consensus&987760-988266&-1,cluster_001_consensus&988275-989306&-1,cluster_001_consensus&994300-994875&-1,...,LNF,cluster_001_consensus&1177640-1178803&-1,LNF,LNF,LNF,LNF,LNF,cluster_001_consensus&584636-585070&-1,cluster_001_consensus&25443-26357&-1,BB520
GCA_000009085,AL111168.1&1-1323&1,AL111168.1&111488-112993&1,AL111168.1&942715-943482&1,AL111168.1&943614-944063&-1,AL111168.1&947343-948038&-1,AL111168.1&948025-948795&-1,AL111168.1&953036-953470&-1,AL111168.1&953723-954229&-1,AL111168.1&954238-955269&-1,AL111168.1&960263-960838&-1,...,LNF,AL111168.1&1109699-1110862&-1,AL111168.1&1373917-1375020&-1,LNF,LNF,LNF,LNF,AL111168.1&550743-551177&-1,AL111168.1&25433-26347&-1,GCA_000009085
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
IFTK99,cluster_001_consensus&1-1323&1,cluster_001_consensus&110122-111627&1,cluster_001_consensus&976450-977217&1,cluster_001_consensus&977349-977813&-1,cluster_001_consensus&981077-981772&-1,cluster_001_consensus&981759-982529&-1,cluster_001_consensus&986770-987204&-1,cluster_001_consensus&987457-987963&-1,cluster_001_consensus&987972-989003&-1,cluster_001_consensus&993997-994572&-1,...,LNF,cluster_001_consensus&1146849-1148012&-1,cluster_001_consensus&1448722-1449828&-1,LNF,LNF,LNF,cluster_001_consensus&1303204-1303995&1,cluster_001_consensus&586298-586732&-1,cluster_001_consensus&23689-24603&-1,IFTK99
LA525,cluster_001_consensus&1-1323&1,cluster_001_consensus&111162-112667&1,cluster_001_consensus&972935-973702&1,cluster_001_consensus&1032103-1032567&-1,cluster_001_consensus&1035831-1036526&-1,cluster_001_consensus&1036513-1037283&-1,cluster_001_consensus&1041524-1041958&-1,cluster_001_consensus&1042211-1042717&-1,cluster_001_c

In [3]:
def path_finder(loci_name, genome_name, tree):
    list_path = []
    phylo_name =str(loci_name+"_")
    for i in range(len(tree.get_path(target=genome_name))):
        if tree.get_path(target=genome_name)[i].branch_length!=0:
            if tree.get_path(target=genome_name)[i].name!=genome_name:
                path_tmp = phylo_name+tree.get_path(target=genome_name)[i].name
                list_path.append(path_tmp)
    return  list_path

In [4]:
PhyloBinary = pd.DataFrame()

for loci_name in tqdm(results_contigsInfo.columns[0:10]):
    Seq_list = []
    for genome in results_contigsInfo.index:
        Info = results_contigsInfo[loci_name][genome].split('&')
        if len(Info) != 1:
            Seqs = SeqIO.parse("genomes/"+results_contigsInfo["GenomeName"][genome]+".fasta", "fasta")
            Seqs = SeqIO.to_dict(Seqs)
            start, end = results_contigsInfo[loci_name][genome].split('&')[1].split('-')
            start, end = int(start)-1, int(end)
            if Info[-1]==str(1):
                Seq_tmp = (genome, Seqs[results_contigsInfo[loci_name][genome].split('&')[0]].seq[start:end])
            if Info[-1]==str(-1):
                Seq_tmp = (genome, Seqs[results_contigsInfo[loci_name][genome].split('&')[0]].seq[start:end].reverse_complement())
            Seq_list.append(Seq_tmp)
    
    if Seq_list!=[]:
        try:
            tree = Phylo.read("alins/output_"+loci_name+".dnd", "newick")
        except ValueError:
            pass
        except FileNotFoundError:
            pass
        else:
            for cnt, node in enumerate(tree.get_nonterminals()):
                if node.name is None:
                    node.name = f"Inner{cnt}"
            df_tmp = pd.DataFrame()
            for k in range(len(results_contigsInfo.index)):
                Info = results_contigsInfo[loci_name][results_contigsInfo.index[k]].split('&')
                if len(Info) != 1:
                    df_tmp.loc[results_contigsInfo.index[k], loci_name] = 1
                    if tree.get_path(target=results_contigsInfo.index[k]) != None:
                        for i in range(len(path_finder(loci_name, results_contigsInfo.index[k], tree))):
                            df_tmp.loc[results_contigsInfo.index[k], path_finder(loci_name, results_contigsInfo.index[k], tree)[i]] = 1


    PhyloBinary = pd.concat([PhyloBinary, df_tmp], axis = 1)
PhyloBinary = PhyloBinary.fillna(0)
PhyloBinary = PhyloBinary.astype('int')

  0%|          | 0/10 [00:00<?, ?it/s]

/var/folders/rf/xn_g934s1372p5c2kg2xn44w0000gn/T/ipykernel_49198/4258991119.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tmp.loc[results_contigsInfo.index[k], path_finder(loci_name, results_contigsInfo.index[k], tree)[i]] = 1
/var/folders/rf/xn_g934s1372p5c2kg2xn44w0000gn/T/ipykernel_49198/4258991119.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tmp.loc[results_contigsInfo.index[k], path_finder(loci_name, results_contigsInfo.index[k], tree)[i]] = 1
/var/folders/rf/xn_g934s1372p5c2kg2xn44w0000gn/T/ipykernel_4

KeyboardInterrupt: 

In [9]:
df_tmp

,UBB527-protein19,UBB527-protein19_Inner1,UBB527-protein19_Inner2,UBB527-protein19_Inner3,UBB527-protein19_Inner4,UBB527-protein19_Inner5,UBB527-protein19_Inner6,UBB527-protein19_Inner7,UBB527-protein19_Inner8,UBB527-protein19_Inner9,...,UBB527-protein19_Inner556,UBB527-protein19_Inner78,UBB527-protein19_Inner128,UBB527-protein19_Inner243,UBB527-protein19_Inner514,UBB527-protein19_Inner570,UBB527-protein19_Inner573,UBB527-protein19_Inner244,UBB527-protein19_Inner103,UBB527-protein19_Inner520
ATCC33250,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ATCC33291,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BB520,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GCA_000009085,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GCA_000011865,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
IFTK96,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
IFTK99,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
LA525,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UBB527,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
PhyloBinary

NameError: name 'PhyloBinary' is not defined

In [ ]:
PhyloBinary = PhyloBinary.loc[:,~PhyloBinary.columns.duplicated()]

In [ ]:
PhyloBinary

In [ ]:
PhyloBinary = pd.DataFrame()

for loci_name in tqdm(results_contigsInfo.columns):
    Seq_list = []
    for genome in results_contigsInfo.index:
        Info = results_contigsInfo[loci_name][genome].split('&')
        if len(Info) != 1:
            Seqs = SeqIO.parse("genomes/"+results_contigsInfo["GenomeName"][genome]+".fasta", "fasta")
            Seqs = SeqIO.to_dict(Seqs)
            start, end = results_contigsInfo[loci_name][genome].split('&')[1].split('-')
            start, end = int(start)-1, int(end)
            if Info[-1]==str(1):
                Seq_tmp = (genome, Seqs[results_contigsInfo[loci_name][genome].split('&')[0]].seq[start:end])
            if Info[-1]==str(-1):
                Seq_tmp = (genome, Seqs[results_contigsInfo[loci_name][genome].split('&')[0]].seq[start:end].reverse_complement())
            Seq_list.append(Seq_tmp)
    
    if Seq_list!=[]:
        try:
            tree = Phylo.read("alins/output_"+loci_name+".dnd", "newick")
        except ValueError:
            pass
        except FileNotFoundError:
            pass
        else:
            for cnt, node in enumerate(tree.get_nonterminals()):
                if node.name is None:
                    node.name = f"Inner{cnt}"
            df_tmp = pd.DataFrame()
            for k in range(len(results_contigsInfo.index)):
                Info = results_contigsInfo[loci_name][results_contigsInfo.index[k]].split('&')
                if len(Info) != 1:
                    df_tmp.loc[results_contigsInfo.index[k], loci_name] = 1
                    if tree.get_path(target=results_contigsInfo.index[k]) != None:
                        for i in range(len(path_finder(loci_name, results_contigsInfo.index[k], tree))):
                            df_tmp.loc[results_contigsInfo.index[k], path_finder(loci_name, results_contigsInfo.index[k], tree)[i]] = 1


    PhyloBinary = pd.concat([PhyloBinary, df_tmp], axis = 1)
PhyloBinary = PhyloBinary.fillna(0)
PhyloBinary = PhyloBinary.astype('int')

In [ ]:
PhyloBinary

In [ ]:
Seq_list

In [ ]:
PhyloBinary = PhyloBinary.loc[:,~PhyloBinary.columns.duplicated()]

In [ ]:
PhyloBinary

In [ ]:
with open('PhyloBinary.pkl', 'wb') as g:
    pickle.dump(PhyloBinary.astype('int'), g)

In [16]:
df_params = pickle.load(open('Parameters sequenced.pkl', 'rb'))
df_params

,delta,power
ATCC33250,2.597162,1.049045
ATCC33291,1.358621,1.298808
ATCC33560,1.804069,1.167914
BB520,1.016268,1.233776
IA525,1.405347,1.572493
IFTK146,1.955503,0.860528
IFTK189,1.841877,1.185209
IFTK197,1.030935,0.845576
IFTK207,1.316486,0.865923
IFTK217,1.422369,0.579437


In [ ]:
PhyloBinary_sequenced = pd.DataFrame()
for s in PhyloBinary.index:
    if s in df_params.index:
        PhyloBinary_sequenced = pd.concat([PhyloBinary_sequenced, PhyloBinary.loc[s].T], axis=1)
    else:
        pass
PhyloBinary_sequenced = PhyloBinary_sequenced.T

In [ ]:
PhyloBinary_sequenced

In [ ]:
with open('PhyloBinary sequenced.pkl', 'wb') as g:
    pickle.dump(PhyloBinary_sequenced, g)

In [ ]:
ProtPhyloBinary = pd.DataFrame()

for loci_name in tqdm(results_contigsInfo.columns):
    Seq_list = []
    for genome in results_contigsInfo.index:
        Info = results_contigsInfo[loci_name][genome].split('&')
        if len(Info) != 1:
            Seqs = SeqIO.parse("genomes/"+results_contigsInfo["GenomeName"][genome]+".fasta", "fasta")
            Seqs = SeqIO.to_dict(Seqs)
            start, end = results_contigsInfo[loci_name][genome].split('&')[1].split('-')
            start, end = int(start)-1, int(end)
            if Info[-1]==str(1):
                Seq_tmp = (genome, Seqs[results_contigsInfo[loci_name][genome].split('&')[0]].seq[start:end].translate(11))
            if Info[-1]==str(-1):
                Seq_tmp = (genome, Seqs[results_contigsInfo[loci_name][genome].split('&')[0]].seq[start:end].reverse_complement().translate(11))
            Seq_list.append(Seq_tmp)
    
    if Seq_list!=[]:
        try:
            tree = Phylo.read("alins/protein/output_"+loci_name+".dnd", "newick")
        except ValueError:
            pass
        except FileNotFoundError:
            pass
        else:
            for cnt, node in enumerate(tree.get_nonterminals()):
                if node.name is None:
                    node.name = f"Inner{cnt}"
            df_tmp = pd.DataFrame()
            for k in range(len(results_contigsInfo.index)):
                Info = results_contigsInfo[loci_name][results_contigsInfo.index[k]].split('&')
                if len(Info) != 1:
                    df_tmp.loc[results_contigsInfo.index[k], loci_name] = 1
                    if tree.get_path(target=results_contigsInfo.index[k]) != None:
                        for i in range(len(path_finder(loci_name, results_contigsInfo.index[k], tree))):
                            df_tmp.loc[results_contigsInfo.index[k], path_finder(loci_name, results_contigsInfo.index[k], tree)[i]] = 1


    ProtPhyloBinary = pd.concat([ProtPhyloBinary, df_tmp], axis = 1)
ProtPhyloBinary = ProtPhyloBinary.fillna(0)
ProtPhyloBinary = ProtPhyloBinary.astype('int')

  0%|          | 0/4187 [00:00<?, ?it/s]

/var/folders/rf/xn_g934s1372p5c2kg2xn44w0000gn/T/ipykernel_49205/933794448.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tmp.loc[results_contigsInfo.index[k], path_finder(loci_name, results_contigsInfo.index[k], tree)[i]] = 1
/var/folders/rf/xn_g934s1372p5c2kg2xn44w0000gn/T/ipykernel_49205/933794448.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_tmp.loc[results_contigsInfo.index[k], path_finder(loci_name, results_contigsInfo.index[k], tree)[i]] = 1
/var/folders/rf/xn_g934s1372p5c2kg2xn44w0000gn/T/ipykernel_492

In [11]:
ProtPhyloBinary

,ATCC33250-protein1,ATCC33250-protein1_Inner1,ATCC33250-protein1_Inner2,ATCC33250-protein1_Inner3,ATCC33250-protein1_Inner4,ATCC33250-protein1_Inner5,ATCC33250-protein1_Inner6,ATCC33250-protein1_Inner7,ATCC33250-protein1_Inner8,ATCC33250-protein1_Inner9,...,UBB527-protein19_Inner556,UBB527-protein19_Inner78,UBB527-protein19_Inner128,UBB527-protein19_Inner243,UBB527-protein19_Inner514,UBB527-protein19_Inner570,UBB527-protein19_Inner573,UBB527-protein19_Inner244,UBB527-protein19_Inner103,UBB527-protein19_Inner520
ATCC33250,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
ATCC33291,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
ATCC33560,1,1,1,1,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
BB520,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
GCA_000009085,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GCA_001587015,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCA_017356945,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCF_001587015,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCF_017356945,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [12]:
ProtPhyloBinary = ProtPhyloBinary.loc[:,~ProtPhyloBinary.columns.duplicated()]

In [13]:
ProtPhyloBinary

,ATCC33250-protein1,ATCC33250-protein1_Inner1,ATCC33250-protein1_Inner2,ATCC33250-protein1_Inner3,ATCC33250-protein1_Inner4,ATCC33250-protein1_Inner5,ATCC33250-protein1_Inner6,ATCC33250-protein1_Inner7,ATCC33250-protein1_Inner8,ATCC33250-protein1_Inner9,...,UBB527-protein19_Inner556,UBB527-protein19_Inner78,UBB527-protein19_Inner128,UBB527-protein19_Inner243,UBB527-protein19_Inner514,UBB527-protein19_Inner570,UBB527-protein19_Inner573,UBB527-protein19_Inner244,UBB527-protein19_Inner103,UBB527-protein19_Inner520
ATCC33250,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
ATCC33291,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
ATCC33560,1,1,1,1,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
BB520,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
GCA_000009085,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GCA_001587015,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCA_017356945,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCF_001587015,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
GCF_017356945,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
with open('ProtPhyloBinary.pkl', 'wb') as g:
    pickle.dump(ProtPhyloBinary.astype('int'), g)

In [17]:
ProtPhyloBinary_sequenced = pd.DataFrame()
for s in ProtPhyloBinary.index:
    if s in df_params.index:
        ProtPhyloBinary_sequenced = pd.concat([ProtPhyloBinary_sequenced, ProtPhyloBinary.loc[s].T], axis=1)
    else:
        pass
ProtPhyloBinary_sequenced = ProtPhyloBinary_sequenced.T

In [18]:
ProtPhyloBinary_sequenced

,ATCC33250-protein1,ATCC33250-protein1_Inner1,ATCC33250-protein1_Inner2,ATCC33250-protein1_Inner3,ATCC33250-protein1_Inner4,ATCC33250-protein1_Inner5,ATCC33250-protein1_Inner6,ATCC33250-protein1_Inner7,ATCC33250-protein1_Inner8,ATCC33250-protein1_Inner9,...,UBB527-protein19_Inner556,UBB527-protein19_Inner78,UBB527-protein19_Inner128,UBB527-protein19_Inner243,UBB527-protein19_Inner514,UBB527-protein19_Inner570,UBB527-protein19_Inner573,UBB527-protein19_Inner244,UBB527-protein19_Inner103,UBB527-protein19_Inner520
ATCC33250,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
ATCC33291,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
ATCC33560,1,1,1,1,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
BB520,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
IA525,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
IFTK146,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
IFTK189,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,1
IFTK197,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
IFTK207,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
IFTK217,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [19]:
with open('ProtPhyloBinary sequenced.pkl', 'wb') as g:
    pickle.dump(ProtPhyloBinary_sequenced, g)